In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ))
else:
    print("GPU is not available to PyTorch.")

In [ ]:
import importlib.metadata as md

packages = [
    "transformers",
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "qwen-vl-utils",
]

for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

In [ ]:
%pip install -q "trl>=0.22.0" "bitsandbytes>=0.45.0" qwen-vl-utils

In [ ]:
import torch
import transformers
import trl
import peft
import bitsandbytes
import qwen_vl_utils

print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=4 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory allocated (GB):",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2))

In [ ]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

print("SFTConfig:", SFTConfig)
print("SFTTrainer:", SFTTrainer)
print("LoraConfig:", LoraConfig)
print("Training classes imported successfully.")

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

print("Datasets attached:")
for p in INPUT_DIR.iterdir():
    print("-", p.name)

for root in INPUT_DIR.iterdir():
    print(f"\nInspecting: {root}")

    for split in ("train", "valid", "test"):
        images_dir = root / split / "images"
        labels_dir = root / split / "labels"

        if images_dir.exists():
            image_count = sum(
                1 for f in images_dir.iterdir()
                if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
            )
            label_count = (
                sum(1 for f in labels_dir.glob("*.txt"))
                if labels_dir.exists() else 0
            )

            print(
                f"{split}: {image_count} images, "
                f"{label_count} label files"
            )

In [ ]:
from pathlib import Path

root = Path("/kaggle/input/datasets")

for p in root.rglob("*"):
    if p.is_dir() and p.name in ("train", "valid", "test"):
        print(f"\nSplit folder: {p}")
        for sub in p.iterdir():
            print(" ", sub.name, "(folder)" if sub.is_dir() else "(file)")

    elif p.is_file() and p.name.lower() in ("data.yaml", "data.yml"):
        print(f"\nYAML file: {p}")
        print(p.read_text())

In [ ]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/musthafa145/"
    "ppe-heuristics/ppe-safety-dataset.zip"
)

for split in ["train", "valid"]:
    images_dir = ROOT / split / "images"
    labels_dir = ROOT / split / "labels"

    images = [
        p for p in images_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]
    labels = list(labels_dir.glob("*.txt"))

    print(f"{split}:")
    print(f"  Images: {len(images)}")
    print(f"  Label files: {len(labels)}")
    print(f"  Images without matching label file: "
          f"{sum(not (labels_dir / (p.stem + '.txt')).exists() for p in images)}")

In [ ]:
from collections import Counter
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/musthafa145/"
    "ppe-heuristics/ppe-safety-dataset.zip"
)

class_names = ["Helmet", "NoHelmet", "NoVest", "Vest"]

for split in ["train", "valid"]:
    counts = Counter()
    empty_labels = 0
    malformed = 0

    for label_file in (ROOT / split / "labels").glob("*.txt"):
        lines = label_file.read_text().splitlines()

        if not lines:
            empty_labels += 1

        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                malformed += 1
                continue

            try:
                class_id = int(parts[0])
                if 0 <= class_id < len(class_names):
                    counts[class_id] += 1
                else:
                    malformed += 1
            except ValueError:
                malformed += 1

    print(f"\n{split.upper()}")
    for i, name in enumerate(class_names):
        print(f"{name}: {counts[i]} objects")
    print("Empty label files:", empty_labels)
    print("Malformed or invalid annotations:", malformed)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path(
    "/kaggle/input/datasets/musthafa145/"
    "ppe-heuristics/ppe-safety-dataset.zip"
)

image_path = next((ROOT / "train" / "images").iterdir())
label_path = ROOT / "train" / "labels" / f"{image_path.stem}.txt"

print("Image:", image_path.name)
print("Annotations:")
print(label_path.read_text())

img = Image.open(image_path).convert("RGB")
plt.figure(figsize=(10, 7))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import json
from PIL import Image

ROOT = Path(
    "/kaggle/input/datasets/musthafa145/"
    "ppe-heuristics/ppe-safety-dataset.zip"
)
OUT = Path("/kaggle/working/ppe_vlm_data")
OUT.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Helmet", "NoHelmet", "NoVest", "Vest"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def convert_split(split):
    images_dir = ROOT / split / "images"
    labels_dir = ROOT / split / "labels"
    records = []
    skipped = []

    image_paths = sorted(
        p for p in images_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTS
    )

    for image_path in image_paths:
        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            skipped.append(image_path.name)
            continue

        objects = []

        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f"Malformed label: {label_path}: {line}")

            class_id = int(parts[0])
            if not 0 <= class_id < len(CLASS_NAMES):
                raise ValueError(f"Invalid class ID in {label_path}: {class_id}")

            x, y, w, h = map(float, parts[1:])
            if not all(0 <= v <= 1 for v in (x, y, w, h)):
                raise ValueError(f"Invalid normalized box in {label_path}: {line}")

            objects.append({
                "class": CLASS_NAMES[class_id],
                "bbox_xywh_normalized": [x, y, w, h],
            })

        with Image.open(image_path) as img:
            width, height = img.size

        records.append({
            "image": str(image_path),
            "prompt": [{
                "role": "user",
                "content": (
                    "Identify every PPE object annotated in this image. "
                    "For each object, report its class and normalized bounding "
                    "box as [center_x, center_y, width, height]. "
                    "Return only a JSON object with an 'objects' list."
                ),
            }],
            "completion": [{
                "role": "assistant",
                "content": json.dumps(
                    {"objects": objects},
                    separators=(",", ":"),
                ),
            }],
            "metadata": {
                "filename": image_path.name,
                "image_width": width,
                "image_height": height,
            },
        })

    return records, skipped

train_records, train_skipped = convert_split("train")
valid_records, valid_skipped = convert_split("valid")

for name, records in [
    ("train", train_records),
    ("valid", valid_records),
]:
    output_file = OUT / f"{name}.jsonl"
    with output_file.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"{name}: {len(records)} records -> {output_file}")

print("Training images skipped:", len(train_skipped))
print("Validation images skipped:", len(valid_skipped))
print("Example completion:")
print(train_records[0]["completion"][0]["content"])

In [ ]:
import json
from pathlib import Path
from PIL import Image

OUT = Path("/kaggle/working/ppe_vlm_data")

for split in ["train", "valid"]:
    file_path = OUT / f"{split}.jsonl"

    with file_path.open(encoding="utf-8") as f:
        records = [json.loads(line) for line in f]

    missing_images = [
        r["image"] for r in records
        if not Path(r["image"]).is_file()
    ]

    print(f"{split}: {len(records)} records")
    print(f"Missing image paths: {len(missing_images)}")

# Inspect one image and its target
sample = records[0]
print("\nSample image:", sample["image"])
print("Prompt:", sample["prompt"][0]["content"])
print("Target:", sample["completion"][0]["content"])

with Image.open(sample["image"]) as img:
    print("Image dimensions:", img.size)

In [ ]:
print("Model class:", type(model).__name__)
print("Processor class:", type(processor).__name__)
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("Quantization config:", getattr(model, "is_loaded_in_4bit", False))

print("\nImage processor:")
print(type(processor.image_processor).__name__)

print("\nChat template available:", processor.chat_template is not None)

In [ ]:
from PIL import Image
import torch

# Load one training image and its prompt
sample = train_records[0]
image = Image.open(sample["image"]).convert("RGB")
prompt = sample["prompt"][0]["content"]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt",
).to(model.device)

print("Image size:", image.size)
print("Input keys:", list(inputs.keys()))
print("Input IDs shape:", inputs["input_ids"].shape)
print("Pixel values shape:", inputs["pixel_values"].shape)
print("Image grid shape:", inputs["image_grid_thw"].shape)

In [ ]:
from peft import LoraConfig, get_peft_model

# Inspect linear-layer names to choose compatible LoRA targets
linear_names = set()

for name, module in model.named_modules():
    if module.__class__.__name__ == "Linear4bit":
        linear_names.add(name.split(".")[-1])

print("4-bit linear layer names:")
print(sorted(linear_names))

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantized base model for adapter training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
print("TRL version:", __import__("trl").__version__)
print("Transformers version:", __import__("transformers").__version__)

import inspect
from trl import SFTTrainer, SFTConfig

print("SFTTrainer accepts processing_class:",
      "processing_class" in inspect.signature(SFTTrainer.__init__).parameters)

print("SFTConfig supports dataset_text_field:",
      "dataset_text_field" in inspect.signature(SFTConfig).parameters)

print("SFTConfig supports max_length:",
      "max_length" in inspect.signature(SFTConfig).parameters)

In [ ]:
sample = train_records[0]

prompt_text = processor.apply_chat_template(
    [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": sample["prompt"][0]["content"]},
            ],
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

target_text = sample["completion"][0]["content"]

print("Prompt template ending:")
print(repr(prompt_text[-200:]))

print("\nTarget answer beginning:")
print(repr(target_text[:300]))

print("\nEOS token:", processor.tokenizer.eos_token)
print("Image token:", processor.image_token if hasattr(processor, "image_token") else "Use model's image token")

In [ ]:
import json
from PIL import Image
import torch

sample = train_records[0]
image = Image.open(sample["image"]).convert("RGB")
target = sample["completion"][0]["content"]

# Build the user portion of the conversation
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": sample["prompt"][0]["content"]},
        ],
    }
]

prompt_text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Append the target and end-of-turn token
full_text = prompt_text + target + processor.tokenizer.eos_token

# Process the image and full sequence
batch = processor(
    text=[full_text],
    images=[image],
    padding=True,
    return_tensors="pt",
).to(model.device)

# Locate the answer boundary in tokenized text
prompt_ids = processor.tokenizer(
    prompt_text, add_special_tokens=False
)["input_ids"]

full_ids = batch["input_ids"][0]
labels = full_ids.clone()
labels[:len(prompt_ids)] = -100
labels[batch["attention_mask"][0] == 0] = -100

print("Input tokens:", full_ids.numel())
print("Prompt tokens:", len(prompt_ids))
print("Supervised answer tokens:", int((labels != -100).sum()))
print("Image tensor:", tuple(batch["pixel_values"].shape))
print("Image grid:", batch["image_grid_thw"].tolist())

In [ ]:
print("Prompt characters:", len(prompt_text))
print("Target characters:", len(target))
print("\nPrompt ending:")
print(repr(prompt_text[-250:]))
print("\nTarget beginning:")
print(repr(target[:500]))
print("\nTarget token count:", len(
    processor.tokenizer(target, add_special_tokens=False)["input_ids"]
))
print("\nFull text ending:")
print(repr(full_text[-150:]))

In [ ]:
import json
from pathlib import Path

for split in ["train", "valid"]:
    path = Path(f"/kaggle/working/ppe_vlm_data/{split}.jsonl")
    lengths = []

    with path.open(encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            target_text = record["completion"][0]["content"]
            lengths.append(
                len(processor.tokenizer(
                    target_text,
                    add_special_tokens=False
                )["input_ids"])
            )

    lengths.sort()
    n = len(lengths)

    print(f"\n{split.upper()} — {n} examples")
    print("Maximum tokens:", lengths[-1])
    print("95th percentile:", lengths[int(0.95 * (n - 1))])
    print("99th percentile:", lengths[int(0.99 * (n - 1))])
    print("Over 1024 tokens:", sum(x > 1024 for x in lengths))
    print("Over 2048 tokens:", sum(x > 2048 for x in lengths))

In [ ]:
from pathlib import Path
import torch

# Dataset files already created in your notebook
TRAIN_JSONL = "/kaggle/working/ppe_vlm_data/train.jsonl"
VALID_JSONL = "/kaggle/working/ppe_vlm_data/valid.jsonl"

# Save the trained LoRA adapter
OUTPUT_DIR = "/kaggle/working/qwen25vl-ppe-lora"

assert Path(TRAIN_JSONL).exists(), f"Missing: {TRAIN_JSONL}"
assert Path(VALID_JSONL).exists(), f"Missing: {VALID_JSONL}"
assert torch.cuda.is_available(), "CUDA GPU is not available"

print("GPU:", torch.cuda.get_device_name(0))
print("Training file:", TRAIN_JSONL)
print("Validation file:", VALID_JSONL)
print("Output directory:", OUTPUT_DIR)
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GiB")

In [ ]:
import json
from pathlib import Path

with open(TRAIN_JSONL, encoding="utf-8") as f:
    first_record = json.loads(next(f))

print("Record keys:", list(first_record.keys()))
print("Image:", first_record.get("image"))
print("Prompt type:", type(first_record.get("prompt")).__name__)
print("Completion type:", type(first_record.get("completion")).__name__)
print("Prompt preview:", str(first_record.get("prompt"))[:300])
print("Completion preview:", str(first_record.get("completion"))[:300])

In [ ]:
import inspect

for name, obj in list(globals().items()):
    if callable(obj) and (
        "collat" in name.lower()
        or "preprocess" in name.lower()
        or "dataset" in name.lower()
    ):
        if getattr(obj, "__module__", "").startswith("__main__"):
            print(f"\n--- {name} ---")
            try:
                print(inspect.getsource(obj)[:4000])
            except (OSError, TypeError):
                print("Source unavailable")

In [ ]:
print("Relevant notebook variables:")

for name in [
    "model", "processor", "train_dataset", "valid_dataset",
    "train_records", "valid_records", "train_data", "valid_data",
    "TRAIN_JSONL", "VALID_JSONL", "OUTPUT_DIR"
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}", end="")
        if hasattr(obj, "__len__"):
            try:
                print(f", length={len(obj)}")
            except Exception:
                print()
        else:
            print()
    else:
        print(f"{name}: not defined")

In [ ]:
import json
from PIL import Image
from torch.utils.data import Dataset

class PPEDataset(Dataset):
    def __init__(self, jsonl_path):
        with open(jsonl_path, "r", encoding="utf-8") as f:
            self.records = [json.loads(line) for line in f]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]

        image = Image.open(record["image"]).convert("RGB")
        prompt = record["prompt"][0]["content"]
        answer = record["completion"][0]["content"]

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": answer},
                ],
            },
        ]

        return {"messages": messages}

train_dataset = PPEDataset(TRAIN_JSONL)
valid_dataset = PPEDataset(VALID_JSONL)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(valid_dataset))
print("First example keys:", list(train_dataset[0].keys()))
print("Image size:", train_dataset[0]["messages"][0]["content"][0]["image"].size)
print("Prompt preview:", train_dataset[0]["messages"][0]["content"][1]["text"][:150])
print("Answer preview:", train_dataset[0]["messages"][1]["content"][0]["text"][:150])

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

print("SFTTrainer signature:")
print(inspect.signature(SFTTrainer.__init__))

print("\nSFTConfig relevant parameters:")
config_params = inspect.signature(SFTConfig.__init__).parameters

for name in [
    "max_length",
    "max_seq_length",
    "dataset_text_field",
    "dataset_kwargs",
    "remove_unused_columns",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "gradient_checkpointing",
    "max_steps",
    "num_train_epochs",
    "fp16",
    "bf16",
    "eval_strategy",
    "evaluation_strategy",
    "save_strategy",
    "logging_steps",
    "report_to",
]:
    if name in config_params:
        print(f"{name}: supported")

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

print("Custom collator supported:",
      "data_collator" in inspect.signature(SFTTrainer.__init__).parameters)

print("Processor supported:",
      "processing_class" in inspect.signature(SFTTrainer.__init__).parameters)

print("Custom dataset preparation supported:",
      "dataset_kwargs" in inspect.signature(SFTConfig.__init__).parameters)

print("Gradient checkpointing supported:",
      "gradient_checkpointing" in inspect.signature(SFTConfig.__init__).parameters)

print("Current model is PEFT-wrapped:",
      hasattr(model, "peft_config"))

print("GPU memory allocated (GB):",
      round(__import__("torch").cuda.memory_allocated() / 1024**3, 2))

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "/kaggle/working/qwen25vl-ppe-lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=20,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to="none",
)

print("Training configuration created.")
print("Maximum steps:", training_args.max_steps)
print("Batch size:", training_args.per_device_train_batch_size)
print("Output:", training_args.output_dir)

In [ ]:
print("Training dataset type:", type(train_dataset).__name__)
print("Validation dataset type:", type(valid_dataset).__name__)

sample = train_dataset[0]
print("Sample keys:", list(sample.keys()))
print("Messages:", len(sample["messages"]))

for i, message in enumerate(sample["messages"]):
    print(f"\nMessage {i} role:", message.get("role"))
    for item in message.get("content", []):
        print("  content type:", item.get("type"))
        if item.get("type") == "text":
            print("  text preview:", item.get("text", "")[:200])
        elif item.get("type") == "image":
            print("  image object:", type(item.get("image")).__name__)

In [ ]:
print("Dataset class:", PPEDataset)
print("Dataset length:", len(train_dataset))
print("Dataset attributes:", vars(train_dataset).keys())

print("\nFirst example:")
sample = train_dataset[0]

for message in sample["messages"]:
    print("\nRole:", message["role"])
    for item in message["content"]:
        if item["type"] == "image":
            print("Image type:", type(item["image"]))
            print("Image size:", item["image"].size)
        elif item["type"] == "text":
            print("Text:", item["text"][:250])

In [ ]:
import torch

def ppe_collator(examples):
    texts = []
    images = []

    for example in examples:
        messages = example["messages"]
        user_msg = messages[0]
        assistant_msg = messages[1]

        image = next(
            item["image"]
            for item in user_msg["content"]
            if item["type"] == "image"
        )
        prompt = next(
            item["text"]
            for item in user_msg["content"]
            if item["type"] == "text"
        )
        answer = next(
            item["text"]
            for item in assistant_msg["content"]
            if item["type"] == "text"
        )

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            },
        ]

        text = processor.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
        images.append(image)

    batch = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()

    # Mask padding tokens.
    labels[batch["attention_mask"] == 0] = -100

    # Mask the user prompt; learn only from the assistant answer.
    for i, example in enumerate(examples):
        messages = example["messages"]
        prompt_conversation = [
            {
                "role": "user",
                "content": messages[0]["content"],
            }
        ]
        prompt_text = processor.apply_chat_template(
            prompt_conversation,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompt_ids = processor.tokenizer(
            prompt_text,
            add_special_tokens=False,
        )["input_ids"]

        labels[i, :len(prompt_ids)] = -100

    batch["labels"] = labels
    return batch


test_batch = ppe_collator([train_dataset[0]])

print("Collator test passed.")
print("Input IDs:", tuple(test_batch["input_ids"].shape))
print("Pixel values:", tuple(test_batch["pixel_values"].shape))
print("Labels:", tuple(test_batch["labels"].shape))
print("Supervised tokens:", int((test_batch["labels"] != -100).sum()))

In [ ]:
batch = test_batch
labels = batch["labels"][0]
input_ids = batch["input_ids"][0]

supervised = (labels != -100).nonzero(as_tuple=True)[0]

print("First supervised token index:", int(supervised[0]))
print("Last supervised token index:", int(supervised[-1]))
print("Supervised token count:", int((labels != -100).sum()))
print("Total sequence length:", len(input_ids))

start = max(0, int(supervised[0]) - 10)
end = min(len(input_ids), int(supervised[0]) + 20)

print("\nTokens around the supervision boundary:")
print(processor.tokenizer.convert_ids_to_tokens(input_ids[start:end].tolist()))

In [ ]:
batch = test_batch

input_ids = batch["input_ids"][0]
labels = batch["labels"][0]

# Print the first 100 tokens with their label status
tokens = processor.tokenizer.convert_ids_to_tokens(
    input_ids[:100].tolist()
)

for i, token in enumerate(tokens):
    status = "TRAIN" if labels[i] != -100 else "MASK"
    print(f"{i:4d} | {status:5s} | {token}")

In [ ]:

tokens = processor.tokenizer.convert_ids_to_tokens(
    test_batch["input_ids"][0].tolist()
)

for i, token in enumerate(tokens):
    if token in ["<|im_start|>", "<|im_end|>", "assistant", "user"]:
        print(i, token)

In [ ]:
def ppe_collator(examples):
    texts, images = [], []

    for example in examples:
        messages = example["messages"]

        conversation = [
            {"role": msg["role"], "content": msg["content"]}
            for msg in messages
        ]

        image = next(
            item["image"]
            for item in messages[0]["content"]
            if item["type"] == "image"
        )

        text = processor.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )

        texts.append(text)
        images.append(image)

    batch = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100

    for i, example in enumerate(examples):
        messages = example["messages"]

        # Build the prompt-only conversation.
        prompt_text = processor.apply_chat_template(
            messages[:1],
            tokenize=False,
            add_generation_prompt=True,
        )

        # Find the prompt boundary in the multimodal sequence.
        prompt_batch = processor(
            text=[prompt_text],
            images=[images[i]],
            padding=False,
            return_tensors="pt",
        )

        prompt_length = prompt_batch["input_ids"].shape[1]

        labels[i, :prompt_length] = -100

    batch["labels"] = labels
    return batch


test_batch = ppe_collator([train_dataset[0]])

tokens = processor.tokenizer.convert_ids_to_tokens(
    test_batch["input_ids"][0].tolist()
)
labels = test_batch["labels"][0]

supervised = (labels != -100).nonzero(as_tuple=True)[0]

print("First supervised index:", int(supervised[0]))
print("Supervised tokens:", int((labels != -100).sum()))
print("Sequence length:", len(tokens))
print("First supervised tokens:")
print(tokens[int(supervised[0]):int(supervised[0]) + 20])

In [ ]:
from datasets import Dataset

train_hf = Dataset.from_list(train_records)
valid_hf = Dataset.from_list(valid_records)

print("Training dataset:", len(train_hf))
print("Validation dataset:", len(valid_hf))
print("Example keys:", train_hf[0].keys())

In [ ]:
from datasets import Dataset
from PIL import Image

def make_vlm_record(record):
    image = Image.open(record["image"]).convert("RGB")

    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": record["prompt"][0]["content"]},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": record["completion"][0]["content"]},
                ],
            },
        ]
    }

train_hf = Dataset.from_list([
    make_vlm_record(r) for r in train_records
])

valid_hf = Dataset.from_list([
    make_vlm_record(r) for r in valid_records
])

print("Training:", len(train_hf))
print("Validation:", len(valid_hf))
print("First example keys:", train_hf[0].keys())
print("First image type:", type(train_hf[0]["messages"][0]["content"][0]["image"]))

In [ ]:
print(train_hf[0]["messages"][0]["content"][0])

In [ ]:
print(train_hf[0]["messages"][0]["content"][1])
print(train_hf[0]["messages"][1]["content"][0]["text"][:200])

In [ ]:
from PIL import Image
import torch

def ppe_collator(examples):
    images, full_texts, prompt_texts = [], [], []

    for ex in examples:
        msg = ex["messages"]
        image_data = msg[0]["content"][0]["image"]

        # Decode image bytes on demand
        if isinstance(image_data, dict) and image_data.get("bytes"):
            import io
            image = Image.open(io.BytesIO(image_data["bytes"])).convert("RGB")
        else:
            image = image_data.convert("RGB")

        images.append(image)

        prompt = msg[0]["content"][1]["text"]
        answer = msg[1]["content"][0]["text"]

        user_msg = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]

        full_msg = user_msg + [{
            "role": "assistant",
            "content": [{"type": "text", "text": answer}],
        }]

        full_texts.append(
            processor.apply_chat_template(
                full_msg, tokenize=False, add_generation_prompt=False
            )
        )
        prompt_texts.append(
            processor.apply_chat_template(
                user_msg, tokenize=False, add_generation_prompt=True
            )
        )

    batch = processor(
        text=full_texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()

    # Mask padding tokens
    labels[batch["attention_mask"] == 0] = -100

    # Mask prompt and image tokens; train only on assistant answers
    for i, (prompt_text, image) in enumerate(zip(prompt_texts, images)):
        prompt_inputs = processor(
            text=[prompt_text],
            images=[image],
            return_tensors="pt",
        )
        prompt_len = prompt_inputs["input_ids"].shape[1]
        labels[i, :prompt_len] = -100

    batch["labels"] = labels
    return batch

In [ ]:
test_batch = ppe_collator([train_hf[0]])

print("Input shape:", test_batch["input_ids"].shape)
print("Label shape:", test_batch["labels"].shape)
print("Supervised tokens:", (test_batch["labels"] != -100).sum().item())

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=valid_hf,
    processing_class=processor,
    data_collator=ppe_collator,
)

print("Trainer created successfully.")

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
print("Model compute dtype:", model.dtype)
print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)

In [ ]:
training_args.fp16 = False
training_args.bf16 = True

In [ ]:
training_args.fp16 = False
training_args.bf16 = False

# Recreate the trainer with the updated settings
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=valid_hf,
    processing_class=processor,
    data_collator=ppe_collator,
)

print("Trainer recreated with FP16 and BF16 disabled.")

In [ ]:
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)
print("Scaler:", trainer.accelerator.scaler)
print("Gradient clipping:", trainer.args.max_grad_norm)

In [ ]:
# Disable gradient scaling and gradient clipping
trainer.accelerator.scaler = None
trainer.args.max_grad_norm = 0.0

print("Scaler:", trainer.accelerator.scaler)
print("Gradient clipping:", trainer.args.max_grad_norm)

In [ ]:
print("Scaler:", trainer.accelerator.scaler)
print("Mixed precision:", trainer.accelerator.mixed_precision)
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)

In [ ]:
import torch
from accelerate.state import AcceleratorState

# Remove the stale accelerator state so a new trainer can initialize cleanly
AcceleratorState._reset_state(reset_partial_state=True)

# Use consistent full-precision Trainer settings
training_args.fp16 = False
training_args.bf16 = False
training_args.max_grad_norm = 1.0

# Clear any leftover gradients from failed attempts
model.zero_grad(set_to_none=True)

print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)
print("Gradient clipping:", training_args.max_grad_norm)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=valid_hf,
    processing_class=processor,
    data_collator=ppe_collator,
)

print("Mixed precision:", trainer.accelerator.mixed_precision)
print("Scaler:", trainer.accelerator.scaler)

In [ ]:
import os, torch, collections

acc = trainer.accelerator
sc = acc.scaler
print("env ACCELERATE_MIXED_PRECISION:", os.environ.get("ACCELERATE_MIXED_PRECISION"))
print("accelerator.mixed_precision:", acc.mixed_precision)
print("scaler:", type(sc).__name__,
      "| enabled:", sc.is_enabled() if sc is not None else None,
      "| scale:", sc.get_scale() if (sc is not None and sc.is_enabled()) else None)
print("args.fp16 / args.bf16:", trainer.args.fp16, trainer.args.bf16)
print("optimizer:", type(trainer.optimizer).__name__ if trainer.optimizer is not None else None)
print("is_bf16_supported:", torch.cuda.is_bf16_supported(), "| capability:", torch.cuda.get_device_capability())

pc = collections.Counter(); gc_ = collections.Counter(); bad = []
for n, p in trainer.model.named_parameters():
    if p.requires_grad:
        pc[str(p.dtype)] += p.numel()
        if p.dtype != torch.float32 and len(bad) < 8: bad.append(n)
        if p.grad is not None: gc_[str(p.grad.dtype)] += p.grad.numel()
print("trainable param numel by dtype:", dict(pc))
print("existing .grad numel by dtype:", dict(gc_))
print("example non-fp32 trainable params:", bad)
print("all-param dtype census:", dict(collections.Counter(str(p.dtype) for p in trainer.model.parameters())))

In [ ]:
import inspect, collections, torch
import trl.trainer.sft_trainer as sft_mod

# A) Evidence: where does TRL cast adapter dtypes?
src = inspect.getsource(sft_mod)
for i, line in enumerate(src.splitlines()):
    if "bfloat16" in line or "casting_to_bf16" in line or "autocast_adapter" in line:
        print(f"{i}: {line.strip()}")

# B) State before the fix
cnt = lambda: dict(collections.Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))
print("trainable dtypes now:", cnt())
print("mixed_precision:", trainer.accelerator.mixed_precision,
      "| scaler:", type(trainer.accelerator.scaler).__name__,
      "| args.fp16/bf16:", trainer.args.fp16, trainer.args.bf16,
      "| optimizer:", trainer.optimizer)

# C) Upcast AFTER construction, BEFORE the optimizer exists
assert trainer.optimizer is None, "optimizer already built with old params; stop and report"
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
    p.grad = None
print("trainable dtypes after upcast:", cnt())
assert set(cnt()) == {"torch.float32"}
assert trainer.accelerator.mixed_precision == "fp16" and trainer.accelerator.scaler is not None

# D) Pilot
trainer.train()
print("peak GiB:", torch.cuda.max_memory_allocated() / 2**30)

In [ ]:
import torch, collections, dataclasses

print("warmup fields:", [f.name for f in dataclasses.fields(SFTConfig) if "warm" in f.name])

full_args = SFTConfig(
    output_dir="/kaggle/working/qwen25vl-ppe-lora",
    num_train_epochs=1, per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=2e-4, warmup_steps=5,
    lr_scheduler_type="cosine", max_steps=100, logging_steps=5,
    save_strategy="steps", save_steps=50, save_total_limit=2, eval_strategy="no",
    fp16=True, bf16=False,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False, dataset_kwargs={"skip_prepare_dataset": True},
    report_to="none",
)
trainer = SFTTrainer(model=model, args=full_args, train_dataset=train_hf,
                     eval_dataset=valid_hf, processing_class=processor,
                     data_collator=ppe_collator)

# Upcast AFTER construction, BEFORE train() (optimizer not built yet)
assert trainer.optimizer is None
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
print("trainable dtypes:", dict(collections.Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad)))
print("mixed_precision:", trainer.accelerator.mixed_precision,
      "| scaler:", type(trainer.accelerator.scaler).__name__,
      "| args.fp16/bf16:", trainer.args.fp16, trainer.args.bf16)

trainer.train()
print("peak GiB:", torch.cuda.max_memory_allocated() / 2**30)

In [ ]:
import torch
bad = [n for n, p in trainer.model.named_parameters()
       if p.requires_grad and not torch.isfinite(p).all()]
print("non-finite trainable tensors:", len(bad), bad[:5])
mx = max(p.abs().max().item() for p in trainer.model.parameters() if p.requires_grad)
print("max |adapter weight|:", mx)
sc = trainer.accelerator.scaler
print("scaler scale:", sc.get_scale() if sc else None)
for h in trainer.state.log_history[6:12]:
    print({k: h.get(k) for k in ("step", "loss", "grad_norm", "learning_rate")})

In [ ]:
import torch
base = model.unload() if hasattr(model, "unload") else None
assert base is not None, "no unload(); restart the runtime and reload"
bad = [n for n, p in base.named_parameters()
       if p.dtype in (torch.float16, torch.bfloat16, torch.float32) and not torch.isfinite(p).all()]
print("non-finite base tensors:", len(bad), bad[:5])

In [ ]:
print([n for n, _ in base.named_modules() if "layers.0." in n and "q_proj" in n][:3])

In [ ]:
from peft import LoraConfig, get_peft_model
lora = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
)
model = get_peft_model(base, lora)
model.print_trainable_parameters()

In [ ]:
import torch, collections
from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback

class StopOnNaN(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        g, l = (logs or {}).get("grad_norm"), (logs or {}).get("loss")
        if (g is not None and not torch.isfinite(torch.tensor(float(g)))) or \
           (l is not None and float(l) > 3 and state.global_step > 10):
            print(f"STOPPING at step {state.global_step}: loss={l}, grad_norm={g}")
            control.should_training_stop = True

args = SFTConfig(
    output_dir="/kaggle/working/qwen25vl-ppe-lora",
    max_steps=60, per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=5e-5, warmup_steps=10,
    lr_scheduler_type="cosine", max_grad_norm=0.3, logging_steps=1,
    save_strategy="no", eval_strategy="no",
    fp16=True, bf16=False,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False, dataset_kwargs={"skip_prepare_dataset": True},
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_hf,
                     eval_dataset=valid_hf, processing_class=processor,
                     data_collator=ppe_collator, callbacks=[StopOnNaN()])

cnt = lambda: dict(collections.Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))
print("dtypes after construction:", cnt())
assert trainer.optimizer is None
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
print("dtypes after upcast:", cnt())
print("mixed_precision:", trainer.accelerator.mixed_precision,
      "| scaler:", type(trainer.accelerator.scaler).__name__,
      "| scale:", trainer.accelerator.scaler.get_scale() if trainer.accelerator.scaler else None)
assert set(cnt()) == {"torch.float32"}

trainer.train()
print("peak GiB:", torch.cuda.max_memory_allocated() / 2**30)
print("scaler scale at end:", trainer.accelerator.scaler.get_scale())

In [ ]:
import os
from pathlib import Path

output_dir = Path("/kaggle/working/qwen25vl-ppe-lora")

checkpoints = sorted(
    output_dir.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1])
) if output_dir.exists() else []

print("Checkpoints found:", len(checkpoints))
for path in checkpoints:
    print(path)

In [ ]:
import os
import torch

for step in [50, 100]:
    path = f"/kaggle/working/qwen25vl-ppe-lora/checkpoint-{step}"
    print(f"\n--- Checkpoint {step} ---")

    if not os.path.isdir(path):
        print("Directory missing")
        continue

    print("Files:", os.listdir(path))

    adapter_path = os.path.join(path, "adapter_model.safetensors")
    if os.path.exists(adapter_path):
        from safetensors import safe_open

        with safe_open(adapter_path, framework="pt", device="cpu") as f:
            tensors = [f.get_tensor(k) for k in f.keys()]
            nonfinite = sum(
                (~torch.isfinite(t)).sum().item() for t in tensors
            )
            total = sum(t.numel() for t in tensors)

        print("Adapter tensors:", len(tensors))
        print("Non-finite values:", nonfinite, "/", total)
    else:
        print("No adapter_model.safetensors file")

In [ ]:

import os
import torch
import collections
from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback

OUTPUT_DIR = "/kaggle/working/qwen25vl-ppe-lora"
STOP_SAVE_DIR = "/kaggle/working/ppe-adapter-at-stop"


class StopOnNaN(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        logs = logs or {}
        g = logs.get("grad_norm")
        l = logs.get("loss")

        if (
            (g is not None and not torch.isfinite(torch.tensor(float(g))))
            or (l is not None and float(l) > 3 and state.global_step > 10)
        ):
            print(
                f"STOPPING at step {state.global_step}: "
                f"loss={l}, grad_norm={g}"
            )
            control.should_training_stop = True
            control.should_save = True


class SaveAdapterOnStop(TrainerCallback):
    def on_train_end(self, args, state, control, **kwargs):
        model_to_save = kwargs["model"]

        os.makedirs(STOP_SAVE_DIR, exist_ok=True)
        model_to_save.save_pretrained(STOP_SAVE_DIR)
        processor.save_pretrained(STOP_SAVE_DIR)

        print(f"Adapter saved at: {STOP_SAVE_DIR}")
        print(f"Training ended at step: {state.global_step}")


args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_steps=60,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    logging_steps=1,

    # Save frequent checkpoints; retain all for recovery.
    save_strategy="steps",
    save_steps=1,
    save_total_limit=None,

    eval_strategy="no",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_hf,
    eval_dataset=valid_hf,
    processing_class=processor,
    data_collator=ppe_collator,
    callbacks=[StopOnNaN(), SaveAdapterOnStop()],
)

cnt = lambda: dict(
    collections.Counter(
        str(p.dtype)
        for p in trainer.model.parameters()
        if p.requires_grad
    )
)

print("Trainable dtypes:", cnt())

assert trainer.optimizer is None, (
    "Optimizer already initialized; use a fresh trainer before changing "
    "trainable parameter dtypes."
)

for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()

print("Trainable dtypes after upcast:", cnt())

print(
    "Mixed precision:", trainer.accelerator.mixed_precision,
    "| Scaler:", type(trainer.accelerator.scaler).__name__,
    "| Scale:",
    trainer.accelerator.scaler.get_scale()
    if trainer.accelerator.scaler is not None else None,
)

assert set(cnt()) == {"torch.float32"}, (
    "Not all trainable parameters are float32."
)

try:
    result = trainer.train()
    print(result)
except Exception:
    # Preserve the current adapter even if training crashes.
    os.makedirs(STOP_SAVE_DIR, exist_ok=True)
    trainer.model.save_pretrained(STOP_SAVE_DIR)
    processor.save_pretrained(STOP_SAVE_DIR)
    print(f"Saved current adapter after failure to: {STOP_SAVE_DIR}")
    raise
finally:
    print(
        "Peak GPU memory (GiB):",
        torch.cuda.max_memory_allocated() / 2**30
    )

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Training complete.")


In [ ]:
import os

ROOT = "/kaggle/working/qwen25vl-ppe-lora"

for root, dirs, files in os.walk(ROOT):
    for d in dirs[:]:
        if d.startswith("checkpoint-"):
            path = os.path.join(root, d)
            print(f"\nCheckpoint: {path}")
            print("Files:", os.listdir(path))

In [ ]:
import os
import torch
from safetensors import safe_open

src = "/kaggle/working/qwen25vl-ppe-lora/checkpoint-10"
dst = "/kaggle/working/ppe-best-step10"

weights = os.path.join(src, "adapter_model.safetensors")

with safe_open(weights, framework="pt", device="cpu") as f:
    total = 0
    bad = 0

    for key in f.keys():
        tensor = f.get_tensor(key)
        total += tensor.numel()
        bad += (~torch.isfinite(tensor)).sum().item()

print("Checkpoint:", src)
print("Adapter parameters:", total)
print("Non-finite values:", bad)

if bad == 0:
    import shutil
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("SAFE COPY CREATED:", dst)
else:
    print("Checkpoint contains invalid values. Do not use it.")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/ppe-best-step10",
    "zip",
    "/kaggle/working/ppe-best-step10"
)

print("ZIP ready: /kaggle/working/ppe-best-step10.zip")

In [ ]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/ppe-best-step10.zip"))

In [ ]:
from IPython.display import HTML, display
import os, base64

path = "/kaggle/working/ppe-best-step10.zip"

print("Exists:", os.path.exists(path))
print("Size (MB):", round(os.path.getsize(path) / (1024 * 1024), 2))

with open(path, "rb") as f:
    data = base64.b64encode(f.read()).decode("ascii")

display(HTML(
    f'<a download="ppe-best-step10.zip" '
    f'href="data:application/zip;base64,{data}">'
    f'Download PPE checkpoint ZIP</a>'
))

In [ ]:
from IPython.display import Javascript, display
from pathlib import Path

path = Path("/kaggle/working/ppe-best-step10.zip")
assert path.exists(), f"File not found: {path}"

# Serve the file through a temporary local HTTP server
import threading
from http.server import HTTPServer, SimpleHTTPRequestHandler
import os

class DownloadHandler(SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory="/kaggle/working", **kwargs)

    def log_message(self, *args):
        pass

server = HTTPServer(("0.0.0.0", 8765), DownloadHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()

print("File ready:", path)
print("Size (MB):", round(path.stat().st_size / 1024**2, 2))
print("Download URL: http://127.0.0.1:8765/ppe-best-step10.zip")

In [ ]:
from IPython.display import Javascript, display
import base64
from pathlib import Path

path = Path("/kaggle/working/ppe-best-step10.zip")

with open(path, "rb") as f:
    encoded = base64.b64encode(f.read()).decode("ascii")

display(Javascript(f"""
const a = document.createElement('a');
a.href = 'data:application/zip;base64,{encoded}';
a.download = 'ppe-best-step10.zip';
a.click();
"""))